# Financial Market Regime & Event Intelligence Engine
## Notebook 01: Exploratory Analysis, Baseline K-Means & Cluster-Count Comparison

Welcome to Notebook 01! In this notebook, we fetch real-world financial market data using `yfinance`, perform initial exploratory data analysis (EDA), engineer quantitative features for the **S&P 500 index (`^GSPC`)**, analyze cross-asset correlations (`TLT` & `GLD`), train a **Baseline K-Means Clustering Model**, and perform a **Cluster-Count Comparison Experiment ($K=2 \dots 6$)** to evaluate model performance empirically.

### Objectives:
1. **Import standard libraries**: `pandas`, `numpy`, `matplotlib`, `plotly`, `yfinance`, and `scikit-learn`.
2. **Download historical market data**: Fetch 5 years of daily price history.
3. **Inspect data structure**: Look at first/last rows, dimensions, column names, summary statistics, and missing values.
4. **Visualize Closing Price**: Create an interactive line chart of S&P 500 closing prices.
5. **Calculate Financial Features**: Compute Daily Return, 20-day Rolling Volatility, 20-day Momentum, and Drawdown.
6. **Cross-Asset Analysis**: Analyze rolling correlations between S&P 500, Treasuries (`TLT`), and Gold (`GLD`).
7. **Baseline Unsupervised ML**: Train a 4-cluster K-Means model on 6 financial features, evaluate silhouette score, and visualize market regimes over time.
8. **Cluster-Count Comparison**: Evaluate K-Means for $K \in \{2, 3, 4, 5, 6\}$ using Silhouette Score and Plotly visualizations.

---
### Step 1: Import Required Libraries

**Why we do this:**
- `pandas`: Data manipulation and DataFrame operations.
- `numpy`: High-performance numerical computations.
- `matplotlib.pyplot`: Base static plotting.
- `plotly.express`: Dynamic interactive charting.
- `yfinance`: Financial data extraction.
- `sklearn.preprocessing.StandardScaler`: Feature standardization.
- `sklearn.cluster.KMeans`: Unsupervised clustering algorithm.
- `sklearn.metrics.silhouette_score`: Cluster quality evaluation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set display options for clean output
pd.set_option('display.max_columns', None)
print("All required libraries successfully imported!")

All required libraries successfully imported!


---
### Step 2: Download Historical Market Data

**Why we do this:**
To analyze market regimes, we need historical price and volume data. We fetch **5 years** (`period="5y"`) of **daily** (`interval="1d"`) data for ticker `^GSPC` (S&P 500 Index).

In [2]:
ticker_symbol = "^GSPC"
sp500_df = yf.download(ticker_symbol, period="5y", interval="1d")

# Flatten MultiIndex columns if returned by yfinance
if isinstance(sp500_df.columns, pd.MultiIndex):
    sp500_df.columns = sp500_df.columns.get_level_values(0)

[*********************100%***********************]  1 of 1 completed

---
### Step 3: Inspect the First and Last Rows

**Why we do this:**
Viewing `head()` and `tail()` confirms data download accuracy and standard fields (`Open`, `High`, `Low`, `Close`, `Volume`).

In [3]:
print("--- First 5 Rows ---")
display(sp500_df.head())

print("\n--- Last 5 Rows ---")
display(sp500_df.tail())

--- First 5 Rows ---


Price,Close,High,Low,Open,Volume
Date,,,,,
2021-09-13,4468.729980,4492.990234,4445.700195,4474.810059,3914220000
2021-09-14,4443.049805,4485.680176,4435.459961,4479.330078,3670460000
2021-09-15,4480.700195,4486.870117,4438.370117,4447.490234,4032020000
2021-09-16,4473.750000,4485.870117,4443.799805,4477.089844,3984560000
2021-09-17,4432.990234,4471.520020,4427.759766,4469.740234,7289530000



--- Last 5 Rows ---


Price,Close,High,Low,Open,Volume
Date,,,,,
2026-09-04,7718.600098,7750.189941,7706.120117,7750.189941,4103570000
2026-09-08,7673.520020,7717.810059,7666.990234,7717.810059,4966930000
2026-09-09,7636.359863,7660.680176,7624.160156,7660.680176,4896940000
2026-09-10,7591.700195,7612.859863,7580.060059,7594.740234,4893500000
2026-09-11,7656.979980,7677.020020,7636.750000,7636.750000,4722480000


---
### Step 4: Check Dimensions (Shape) and Column Names

**Why we do this:**
- `shape`: Confirms the number of trading days (rows) and attributes (columns).
- `columns`: Displays exact field names.

In [4]:
print(f"DataFrame Shape (Rows, Columns): {sp500_df.shape}")
print("Column Names:", list(sp500_df.columns))

DataFrame Shape (Rows, Columns): (1255, 5)
Column Names: ['Close', 'High', 'Low', 'Open', 'Volume']


---
### Step 5: Basic Descriptive Statistics

**Why we do this:**
`describe()` provides a statistical overview of numerical columns (mean, std dev, min, max, percentiles).

In [5]:
display(sp500_df.describe())

Price,Close,High,Low,Open,Volume
count,1255.000000,1255.000000,1255.000000,1255.000000,1.255000e+03
mean,5285.028853,5313.134363,5253.265982,5284.099399,4.538619e+09
std,1150.139442,1150.576624,1148.918739,1149.908197,1.029365e+09
min,3577.030029,3608.340088,3491.580078,3520.370117,0.000000e+00
25%,4305.729980,4340.819824,4272.274902,4311.750000,3.842920e+09
50%,5069.759766,5097.660156,5039.830078,5074.600098,4.368380e+09
75%,6091.215088,6111.664795,6073.784912,6086.079834,5.098625e+09
max,7798.990234,7816.700195,7776.310059,7806.600098,1.002582e+10


---
### Step 6: Check for Missing Values

**Why we do this:**
Missing values (`NaN`) must be identified before applying mathematical calculations.

In [6]:
print("Missing values per column:")
print(sp500_df.isnull().sum())

Missing values per column:
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


---
### Step 7: Visualize S&P 500 Closing Price Over Time

**Why we do this:**
Visualizing closing prices gives an immediate perspective on market trends over the 5-year timeline.

In [7]:
plot_df = sp500_df.reset_index()

fig_close = px.line(
    plot_df,
    x="Date",
    y="Close",
    title="S&P 500 Index (^GSPC) - 5-Year Closing Price History",
    labels={"Date": "Date", "Close": "S&P 500 Close Price (USD)"},
    template="plotly_white"
)

fig_close.update_layout(title_x=0.5, hovermode="x unified")
fig_close.show()

---
### Step 8: Financial Feature Explanations & Formulas

In financial market regime analysis, raw prices alone are insufficient. We engineer specific quantitative features:

1. **Daily Return (`Daily_Return`)**:
   - **Meaning**: The fractional price change from the previous trading day.
   - **Formula**: `Daily_Return[t] = (Close[t] - Close[t-1]) / Close[t-1]`

2. **20-Day Rolling Volatility (`Rolling_Volatility_20`)**:
   - **Meaning**: Standard deviation of daily returns over a rolling 20-trading-day window (~1 calendar month). Indicates recent market risk and turbulence.
   - **Formula**: `Rolling_Volatility_20[t] = std(Daily_Return[t-19 : t])`

3. **20-Day Momentum (`Momentum_20`)**:
   - **Meaning**: The percentage price change over the past 20 trading days (~1 calendar month). Measures trend strength and directional velocity.
   - **Formula**: `Momentum_20[t] = (Close[t] - Close[t-20]) / Close[t-20]`

4. **Drawdown (`Drawdown`)**:
   - **Meaning**: The percentage decline from the highest peak achieved up to date `t`. Measures peak-to-trough market losses.
   - **Formula**: `Running_Peak[t] = max(Close[0 : t])`, `Drawdown[t] = (Close[t] - Running_Peak[t]) / Running_Peak[t]`

---
### Step 9: Feature Calculations & Inspecting Last 10 Rows

**Why we do this:**
We compute the feature columns directly on our DataFrame `sp500_df` and inspect the latest values.

In [8]:
# Calculate Daily Return
sp500_df["Daily_Return"] = sp500_df["Close"].pct_change()

# 1. 20-day rolling volatility
sp500_df["Rolling_Volatility_20"] = sp500_df["Daily_Return"].rolling(window=20).std()

# 2. 20-day momentum
sp500_df["Momentum_20"] = sp500_df["Close"].pct_change(periods=20)

# 3. Drawdown from historical running peak
running_peak = sp500_df["Close"].cummax()
sp500_df["Drawdown"] = (sp500_df["Close"] - running_peak) / running_peak

# Display required columns for the last 10 rows
feature_columns = ["Close", "Daily_Return", "Rolling_Volatility_20", "Momentum_20", "Drawdown"]
print("--- Last 10 Rows with Financial Features ---")
display(sp500_df[feature_columns].tail(10))

--- Last 10 Rows with Financial Features ---


Price,Close,Daily_Return,Rolling_Volatility_20,Momentum_20,Drawdown
Date,,,,,
2026-08-28,7711.759766,-0.002487,0.006694,0.029646,-0.011185
2026-08-31,7686.140137,-0.003322,0.005987,0.011268,-0.014470
2026-09-01,7631.470215,-0.007113,0.004639,-0.013578,-0.021480
2026-09-02,7666.600098,0.004603,0.004778,-0.007374,-0.016975
2026-09-03,7747.709961,0.010580,0.005350,0.004896,-0.006575
2026-09-04,7718.600098,-0.003757,0.005231,-0.005032,-0.010308
2026-09-08,7673.520020,-0.005840,0.005379,-0.010266,-0.016088
2026-09-09,7636.359863,-0.004843,0.005435,-0.011884,-0.020853
2026-09-10,7591.700195,-0.005848,0.005501,-0.020236,-0.026579


---
### Step 10: Visualize 20-Day Rolling Volatility Over Time

**Why we do this:**
Rolling volatility spikes during market panics or crises. Plotting this feature helps identify high-risk market regimes.

In [9]:
plot_df = sp500_df.reset_index()

fig_vol = px.line(
    plot_df,
    x="Date",
    y="Rolling_Volatility_20",
    title="S&P 500 Index (^GSPC) - 20-Day Rolling Volatility",
    labels={"Date": "Date", "Rolling_Volatility_20": "20-Day Rolling Volatility (Std Dev)"},
    template="plotly_white"
)

fig_vol.update_layout(title_x=0.5, hovermode="x unified")
fig_vol.show()

---
### Step 11: Visualize Historical Drawdown Over Time

**Why we do this:**
Drawdown shows the magnitude of market pullbacks from all-time highs, helping us classify market correction vs. bull expansion regimes.

In [10]:
fig_dd = px.line(
    plot_df,
    x="Date",
    y="Drawdown",
    title="S&P 500 Index (^GSPC) - Historical Drawdown from Peak",
    labels={"Date": "Date", "Drawdown": "Drawdown (Fraction from Peak)"},
    template="plotly_white"
)

fig_dd.update_layout(title_x=0.5, hovermode="x unified")
fig_dd.show()

---
### Step 12: Cross-Asset Correlation Analysis - Concept & Regime Significance

**What is Cross-Asset Correlation?**
Cross-asset correlation measures how different asset classes (e.g., Equities, Government Bonds, Gold) move in relation to one another. Correlation ranges between `-1.0` and `+1.0`:
- `+1.0`: Perfect positive correlation (assets move in lockstep).
- `0.0`: No linear relationship (assets move independently).
- `-1.0`: Perfect negative correlation (assets move in opposite directions).

**Why Cross-Asset Correlation Identifies Market Regimes:**
1. **Risk-On Regime**: Equities (`S&P 500`) rise while safe-haven assets like Long-Term Treasuries (`TLT`) or Gold (`GLD`) often experience muted gains or inverse movements.
2. **Flight-to-Safety / Crisis Regime**: During stock crashes, investors panic and buy Treasuries or Gold, creating strong negative correlations between stocks and safe havens.
3. **Inflation / Monetary Tightening Regime**: When central banks raise interest rates, both stocks and long-term bonds can fall simultaneously (turning stock-bond correlation positive).

Tracking dynamic **20-day rolling correlations** reveals macro regime shifts in real time.

---
### Step 13: Download Cross-Asset Data & Calculate Daily Returns

We download 5 years of daily data for three major asset classes:
- `^GSPC` (S&P 500 - Equities)
- `TLT` (iShares 20+ Year Treasury Bond ETF - Government Bonds)
- `GLD` (SPDR Gold Shares ETF - Gold / Commodity Safe-Haven)

In [11]:
# Download closing prices for each instrument
sp500_series = yf.download("^GSPC", period="5y", interval="1d")["Close"]
tlt_series = yf.download("TLT", period="5y", interval="1d")["Close"]
gld_series = yf.download("GLD", period="5y", interval="1d")["Close"]

# Ensure Series format
if isinstance(sp500_series, pd.DataFrame): sp500_series = sp500_series.squeeze()
if isinstance(tlt_series, pd.DataFrame): tlt_series = tlt_series.squeeze()
if isinstance(gld_series, pd.DataFrame): gld_series = gld_series.squeeze()

# Combine daily percentage returns into a single DataFrame
cross_asset_df = pd.DataFrame({
    "SP500_Return": sp500_series.pct_change(),
    "TLT_Return": tlt_series.pct_change(),
    "GLD_Return": gld_series.pct_change()
})

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

---
### Step 14: Calculate 20-Day Rolling Correlations & Inspect Last 10 Rows

We compute rolling 20-day correlations:
1. `SP500_TLT_Corr_20`: Rolling 20-day correlation between S&P 500 returns and TLT returns.
2. `SP500_GLD_Corr_20`: Rolling 20-day correlation between S&P 500 returns and GLD returns.

In [12]:
# Calculate 20-day rolling correlations
cross_asset_df["SP500_TLT_Corr_20"] = cross_asset_df["SP500_Return"].rolling(window=20).corr(cross_asset_df["TLT_Return"])
cross_asset_df["SP500_GLD_Corr_20"] = cross_asset_df["SP500_Return"].rolling(window=20).corr(cross_asset_df["GLD_Return"])

print("--- Last 10 Rows of Cross-Asset Return & Correlation DataFrame ---")
display(cross_asset_df.tail(10))

--- Last 10 Rows of Cross-Asset Return & Correlation DataFrame ---


,SP500_Return,TLT_Return,GLD_Return,SP500_TLT_Corr_20,SP500_GLD_Corr_20
Date,,,,,
2026-08-28,-0.002487,-0.003007,-0.032442,0.413911,0.120450
2026-08-31,-0.003322,-0.004344,-0.001149,0.434835,0.178162
2026-09-01,-0.007113,-0.004075,-0.028574,0.388198,0.320467
2026-09-02,0.004603,0.000977,0.015198,0.394444,0.417429
2026-09-03,0.010580,0.001464,0.018472,0.371187,0.458907
2026-09-04,-0.003757,0.001706,-0.008410,0.342103,0.430574
2026-09-08,-0.005840,-0.000122,-0.017332,0.347597,0.466429
2026-09-09,-0.004843,-0.005718,0.009081,0.380929,0.431876
2026-09-10,-0.005848,-0.011624,-0.017330,0.431073,0.447687


---
### Step 15: Visualize S&P 500 vs TLT Rolling 20-Day Correlation Over Time

In [13]:
plot_cross_df = cross_asset_df.reset_index()

fig_tlt_corr = px.line(
    plot_cross_df,
    x="Date",
    y="SP500_TLT_Corr_20",
    title="S&P 500 vs. Long-Term Treasuries (TLT) - 20-Day Rolling Correlation",
    labels={"Date": "Date", "SP500_TLT_Corr_20": "20-Day Rolling Correlation"},
    template="plotly_white"
)

fig_tlt_corr.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Zero Correlation Line")
fig_tlt_corr.update_layout(title_x=0.5, hovermode="x unified")
fig_tlt_corr.show()

---
### Step 16: Visualize S&P 500 vs GLD Rolling 20-Day Correlation Over Time

In [14]:
fig_gld_corr = px.line(
    plot_cross_df,
    x="Date",
    y="SP500_GLD_Corr_20",
    title="S&P 500 vs. Gold ETF (GLD) - 20-Day Rolling Correlation",
    labels={"Date": "Date", "SP500_GLD_Corr_20": "20-Day Rolling Correlation"},
    template="plotly_white"
)

fig_gld_corr.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Zero Correlation Line")
fig_gld_corr.update_layout(title_x=0.5, hovermode="x unified")
fig_gld_corr.show()

---
### Step 17: Baseline Unsupervised ML Experiment - K-Means Clustering

#### Conceptual Overview & ML Principles:

1. **Why We Standardize Features (`StandardScaler`)**:
   - Features live on vastly different numerical scales (e.g., `Daily_Return` ranges around `~0.01`, whereas `Drawdown` can be `-0.25`, and correlations range from `-1.0` to `+1.0`).
   - K-Means uses **Euclidean Distance** to compute proximity between data points. Without standardization, features with larger magnitudes would artificially dominate distance calculations.
   - `StandardScaler` rescales every feature to have a **mean of 0** and a **variance/standard deviation of 1** (z = (x - mean) / std).

2. **What K-Means Does Conceptually**:
   - K-Means partitions the 6-dimensional feature space into K=4 distinct geometric clusters.
   - It iteratively places 4 cluster centroids and assigns each trading day to the nearest centroid, minimizing the within-cluster sum of squared distances.

3. **Why Cluster Numbers (0, 1, 2, 3) Do NOT Inherently Mean 'Bull' or 'Bear'**:
   - K-Means is purely mathematical and unsupervised. It assigns integer cluster IDs (`0`, `1`, `2`, `3`) based on initial random seed locations (`random_state=42`).
   - Cluster `0` might represent high volatility in one run and low volatility in another. We must analyze the cluster feature means to interpret the financial meaning of each cluster.

4. **What the Silhouette Score Measures**:
   - The **Silhouette Score** ranges from `-1` to `+1`:
     - **+1**: Observations are extremely well clustered and far from neighboring clusters.
     - **0**: Observations lie right on the decision boundary between clusters.
     - **-1**: Observations are assigned to the wrong cluster.
   - It quantifies how compact and distinct our baseline market clusters are.

---
### Step 18: Build Feature Matrix & Handle Incomplete Rows

We extract our six selected ML features:
- `Daily_Return`
- `Rolling_Volatility_20`
- `Momentum_20`
- `Drawdown`
- `SP500_TLT_Corr_20`
- `SP500_GLD_Corr_20`

We drop the initial `NaN` rows caused by the 20-day rolling window while preserving the original `Date` index.

In [15]:
# List of 6 target ML features
ml_feature_names = [
    "Daily_Return",
    "Rolling_Volatility_20",
    "Momentum_20",
    "Drawdown",
    "SP500_TLT_Corr_20",
    "SP500_GLD_Corr_20"
]

# Combine features into a single DataFrame aligned by Date
ml_raw_df = pd.concat([
    sp500_df[["Daily_Return", "Rolling_Volatility_20", "Momentum_20", "Drawdown"]],
    cross_asset_df[["SP500_TLT_Corr_20", "SP500_GLD_Corr_20"]]
], axis=1)

# Drop incomplete NaN rows from rolling windows, preserving Date index
ml_clean_df = ml_raw_df[ml_feature_names].dropna().copy()

print(f"Cleaned ML Feature Dataset Shape: {ml_clean_df.shape}")

Cleaned ML Feature Dataset Shape: (1235, 6)


---
### Step 19: Fit K-Means Model, Calculate Silhouette Score & Inspect Cluster Stats

We fit K-Means with `n_clusters=4`, `random_state=42`, `n_init="auto"`.

In [16]:
# 1. Standardize features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(ml_clean_df[ml_feature_names])

# 2. Fit K-Means Clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init="auto")
cluster_labels = kmeans.fit_predict(X_scaled)

# 3. Add cluster label to dataset
ml_clean_df["Regime_Cluster"] = cluster_labels

# 4. Calculate Silhouette Score
sil_score = silhouette_score(X_scaled, cluster_labels)
print(f"=== K-Means Clustering Baseline Evaluation ===")
print(f"Silhouette Score (K=4): {sil_score:.4f}\n")

# 5. Display first 10 rows
print("--- First 10 Rows of ML Dataset with Regime Cluster ---")
display(ml_clean_df.head(10))

# 6. Display observations per cluster
print("\n--- Number of Observations per Cluster ---")
display(ml_clean_df["Regime_Cluster"].value_counts().sort_index())

# 7. Display mean of each original feature per cluster
print("\n--- Feature Means per Cluster (Original Scale) ---")
display(ml_clean_df.groupby("Regime_Cluster")[ml_feature_names].mean())

=== K-Means Clustering Baseline Evaluation ===
Silhouette Score (K=4): 0.1848

--- First 10 Rows of ML Dataset with Regime Cluster ---


,Daily_Return,Rolling_Volatility_20,Momentum_20,Drawdown,SP500_TLT_Corr_20,SP500_GLD_Corr_20,Regime_Cluster
Date,,,,,,,
2021-10-11,-0.006866,0.009750,-0.024065,-0.026672,-0.135905,-0.322120,3
2021-10-12,-0.002417,0.009696,-0.020797,-0.029025,-0.105189,-0.308855,3
2021-10-13,0.003023,0.009490,-0.026090,-0.026090,-0.067239,-0.207522,3
2021-10-14,0.017063,0.010337,-0.007933,-0.009472,-0.014515,-0.222060,1
2021-10-15,0.007460,0.010262,0.008658,-0.002082,-0.049185,-0.288505,1
2021-10-18,0.003375,0.009413,0.029541,0.000000,0.101646,-0.240841,1
2021-10-19,0.007393,0.009486,0.037996,0.000000,0.060426,-0.218240,1
2021-10-20,0.003664,0.009328,0.031975,0.000000,0.018577,-0.185849,1
2021-10-21,0.002996,0.009003,0.022657,0.000000,0.173976,-0.108886,1



--- Number of Observations per Cluster ---


Regime_Cluster
0    454
1    280
2    202
3    299
Name: count, dtype: int64


--- Feature Means per Cluster (Original Scale) ---


,Daily_Return,Rolling_Volatility_20,Momentum_20,Drawdown,SP500_TLT_Corr_20,SP500_GLD_Corr_20
Regime_Cluster,,,,,,
0,0.000237,0.007481,0.014809,-0.024370,0.332657,0.403037
1,0.002268,0.007788,0.042757,-0.037336,0.096319,-0.187537
2,-0.001624,0.016769,-0.031875,-0.174945,0.192487,0.360893
3,0.000712,0.010582,0.000349,-0.067414,-0.291982,0.001078


---
### Step 20: Visualize Regime Clusters Over Time

Plotting the assigned `Regime_Cluster` index over time illustrates how the baseline model transitions between regime states.

In [17]:
plot_ml_df = ml_clean_df.reset_index()

fig_regimes_ts = px.line(
    plot_ml_df,
    x="Date",
    y="Regime_Cluster",
    title="S&P 500 Market Regime Clusters Over Time (K-Means K=4)",
    labels={"Date": "Date", "Regime_Cluster": "Cluster Label (0, 1, 2, 3)"},
    template="plotly_white"
)

fig_regimes_ts.update_traces(mode="lines+markers", marker=dict(size=4))
fig_regimes_ts.update_layout(title_x=0.5, yaxis=dict(dtick=1))
fig_regimes_ts.show()

---
### Step 21: Visualize Feature Space (Momentum vs Volatility Scatter Plot)

We plot 20-Day Momentum against 20-Day Rolling Volatility, color-coded by `Regime_Cluster`, to visually inspect cluster separation in 2D space.

In [18]:
# Convert cluster integer to categorical string for distinct legend colors
plot_ml_df["Regime_Cluster_Label"] = "Cluster " + plot_ml_df["Regime_Cluster"].astype(str)

fig_scatter = px.scatter(
    plot_ml_df,
    x="Momentum_20",
    y="Rolling_Volatility_20",
    color="Regime_Cluster_Label",
    title="Baseline K-Means Regimes: 20-Day Momentum vs. 20-Day Rolling Volatility",
    labels={
        "Momentum_20": "20-Day Momentum",
        "Rolling_Volatility_20": "20-Day Rolling Volatility",
        "Regime_Cluster_Label": "Regime Cluster"
    },
    template="plotly_white"
)

fig_scatter.update_layout(title_x=0.5)
fig_scatter.show()

---
### Step 22: Cluster-Count (K) Comparison Experiment

#### Why Compare Different Values of K?
1. **Baseline Choice vs. Empirical Comparison**:
   - K=4 was initially chosen as an intuitive starting baseline for exploration (e.g., potential regimes: Bull, Bear, High Volatility, Crisis).
   - We are now systematically evaluating K in {2, 3, 4, 5, 6} using the cleaned 6-feature dataset and standardized scaling.

2. **Understanding Silhouette Score Trade-offs**:
   - A higher Silhouette Score indicates tighter, more geometrically distinct clusters.
   - However, **Silhouette Score alone does not dictate the financially correct number of regimes**. A lower K (e.g., K=2) might yield a higher geometric score by simply separating low vs. high volatility, but it may fail to capture nuanced financial market dynamics like stagflation or momentum corrections.

3. **Experimental Principles**:
   - We leave the baseline K=4 results intact.
   - We do not rename clusters or fit HMM models yet.

---
### Step 23: Fit K-Means for K = 2, 3, 4, 5, 6 & Display Results Table

In [19]:
k_values = [2, 3, 4, 5, 6]
k_results = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    k_results.append({"K": k, "Silhouette_Score": round(score, 4)})

k_comparison_df = pd.DataFrame(k_results)

print("--- K-Means Cluster Count Comparison (K = 2 to 6) ---")
display(k_comparison_df)

--- K-Means Cluster Count Comparison (K = 2 to 6) ---


,K,Silhouette_Score
0,2,0.2869
1,3,0.2190
2,4,0.1848
3,5,0.1565
4,6,0.1717


---
### Step 24: Visualize Silhouette Score vs. K

In [20]:
fig_k_comp = px.line(
    k_comparison_df,
    x="K",
    y="Silhouette_Score",
    markers=True,
    title="K-Means Clustering: Silhouette Score vs. Number of Clusters (K)",
    labels={"K": "Number of Clusters (K)", "Silhouette_Score": "Silhouette Score"},
    template="plotly_white"
)

fig_k_comp.update_layout(title_x=0.5, xaxis=dict(dtick=1))
fig_k_comp.show()